# Data Analysis Project Phase 1: Data Collection and MongoDB
**Name:** Austin Dachel

## Project Summary
This program downloads crash data in JSON format via HTTP, writes the data to MongoDB Atlas, saves the downloaded JSON to a local file, and verifies the number of records saved.

The original data source is NHTSA CrashAPI (FARS fatal crash-related data). Due to API access limits, the JSON files were mirrored into a GitHub repository and downloaded from GitHub raw URLs via HTTP.

## Configuration and Setup

- MongoDB Atlas connection information
- Database and collection names
- GitHub mirror repository location
- https://www.github.com/ausdac/crash-data-mirror/tree/main
- Years included in the dataset (2019–2023)
- File naming patterns for crash-level and vehicle-level JSON files
- Local directory where downloaded JSON files will be stored

In [1]:
import os
from pathlib import Path

# Mongo information & auth
MONGO_URI = os.getenv(
    "MONGO_URI",
    "mongodb+srv://austindachel_db_user:<redacted>@coloradomotofatalities.8s8a8nt.mongodb.net/?appName=ColoradoMotoFatalities"
)

# Database + collection names
DB_NAME = "ColoradoMotoFatalities"   
COLL_CRASHES = "co_fatal_crashes_2019_2023"
COLL_VEHICLES = "co_vehicles_2019_2023"

# Github mirror repo information
OWNER = "ausdac"
REPO = "crash-data-mirror"
BRANCH = "main"
DATA_DIR = "data"

# Years included for pulling data
YEARS = [2019, 2020, 2021, 2022, 2023]

# File naming pattern for the repo
CRASH_FILE = "colorado_fatal_crashes_{year}.json" # Crash level dataset
VEH_FILE   = "colorado_motorcycle_{year}.json" # Vehicle level dataset

# Write to a local file
OUT_DIR = Path("downloaded_json")
OUT_DIR.mkdir(exist_ok=True)


## Downloading JSON Data via HTTP

1. Builds raw GitHub URLs for each JSON file.
2. Downloads each file using the requests library.
3. Saves the downloaded content locally.
4. Computes file metadata (size, record count, SHA-256 hash).
5. Normalizes the JSON structure into a list of records.
6. Prints dataset summary statistics.

Data Collected:
- Fatal crash-level data (2019–2023)
- Motorcycle vehicle-level data (2019–2023)

In [2]:
import json
import hashlib
from datetime import datetime, timezone
import requests

# Raw github URL for the files stored in the mirror repository
def raw_url(filename: str) -> str:
    return f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/{DATA_DIR}/{filename}"

# Added in a compute for a SHA-256 checksum for the downloadad file
# Source: https://docs.python.org/3/library/hashlib.html
def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

# Normalize the different jsons into a simple list
def normalize_rows(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict) and "Results" in payload:
        res = payload["Results"]
        if res and isinstance(res[0], list):
            return res[0]
        return res
    raise TypeError(f"Unexpected JSON shape: {type(payload).__name__}")

# Download one json file, save it locally and return:
# data
# payload
# rows
def download_and_save(filename: str) -> dict:
    url = raw_url(filename)

    # json download
    r = requests.get(url, timeout=60)
    r.raise_for_status()

    content = r.content
    out_path = OUT_DIR / filename
    out_path.write_bytes(content)

    # parse json and normalize records
    payload = r.json()
    rows = normalize_rows(payload)

    # build metadata for reporting
    meta = {
        "filename": filename,
        "url": url,
        "saved_to": str(out_path),
        "bytes": len(content),
        "sha256": sha256_bytes(content),
        "records": len(rows),
        "downloadedAtUtc": datetime.now(timezone.utc).isoformat(),
    }
    return {"meta": meta, "payload": payload, "rows": rows}

# download all files
downloads = []
for y in YEARS:
    downloads.append(download_and_save(CRASH_FILE.format(year=y)))
    downloads.append(download_and_save(VEH_FILE.format(year=y)))

# print a quick dataset summary
for d in downloads:
    m = d["meta"]
    print(f'{m["filename"]}: records={m["records"]}, sizeMB={m["bytes"]/1024/1024:.2f}')


colorado_fatal_crashes_2019.json: records=543, sizeMB=0.09
colorado_motorcycle_2019.json: records=872, sizeMB=4.58
colorado_fatal_crashes_2020.json: records=570, sizeMB=0.09
colorado_motorcycle_2020.json: records=885, sizeMB=4.78
colorado_fatal_crashes_2021.json: records=637, sizeMB=0.10
colorado_motorcycle_2021.json: records=1024, sizeMB=5.52
colorado_fatal_crashes_2022.json: records=699, sizeMB=0.11
colorado_motorcycle_2022.json: records=1089, sizeMB=5.84
colorado_fatal_crashes_2023.json: records=664, sizeMB=0.11
colorado_motorcycle_2023.json: records=1010, sizeMB=5.47


In [3]:
# write a single json file containing all downloaded data
all_out = OUT_DIR / "co_phase1_all_downloads.json"
all_out.write_text(
    json.dumps(
        [{"meta": d["meta"], "payload": d["payload"]} for d in downloads],
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)
print("Wrote:", all_out)


Wrote: downloaded_json/co_phase1_all_downloads.json


## Writing Data to MongoDB Atlas

1. Connects to MongoDB Atlas using pymongo.
2. Clears the collections to allow reruns of the notebook.
3. Inserts crash-level records into:
   - co_fatal_crashes_2019_2023
4. Inserts vehicle-level records into:
   - co_vehicles_2019_2023
5. Adds a year field to each document for future phases

In [4]:
from pymongo import MongoClient

# connect to mongoDB atlas
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
crashes = db[COLL_CRASHES]
vehicles = db[COLL_VEHICLES]

# Make it rerunnable by clearing old data to avoid duplicates
crashes.delete_many({})
vehicles.delete_many({})

crash_inserted = 0
veh_inserted = 0

# insert into mongo
for d in downloads:
    filename = d["meta"]["filename"]
    rows = d["rows"]
    year = int(filename.split("_")[-1].split(".")[0])

    # extract year from the filename suffix
    if filename.startswith("colorado_fatal_crashes_"):
        # add a year field so its easier to query
        for r in rows:
            r["year"] = year
        if rows:
            crashes.insert_many(rows)
            crash_inserted += len(rows)

    elif filename.startswith("colorado_motorcycle_"):
        for r in rows:
            r["year"] = year
        if rows:
            vehicles.insert_many(rows)
            veh_inserted += len(rows)

print("Inserted crashes:", crash_inserted)
print("Inserted vehicles:", veh_inserted)

# Verification
print("\nCounts in MongoDB:")
print("Crash docs:", crashes.count_documents({}))
print("Vehicle docs:", vehicles.count_documents({}))

print("\nExample crash document:")
print(crashes.find_one({}, {"_id": 0}))

print("\nExample vehicle document:")
print(vehicles.find_one({}, {"_id": 0, "ST_CASE": 1, "VEH_NO": 1, "BODY_TYP": 1, "BODY_TYPNAME": 1, "DEATHS": 1, "year": 1}))


Inserted crashes: 3113
Inserted vehicles: 4880

Counts in MongoDB:
Crash docs: 3113
Vehicle docs: 4880

Example crash document:
{'CountyName': 'ADAMS (1)', 'CrashDate': '/Date(1546518000000-0500)/', 'Fatals': 1, 'Peds': 0, 'Persons': 1, 'St_Case': 80002, 'State': 8, 'StateName': 'Colorado', 'TotalVehicles': 1, 'year': 2019}

Example vehicle document:
{'BODY_TYP': '84', 'BODY_TYPNAME': 'Motor Scooter', 'DEATHS': '1', 'ST_CASE': '80001', 'VEH_NO': '1', 'year': 2019}
